# 3.1.4 — Cartan ilişkilerinden türetilen üç özdeşlik (standart + tilde)

**Hedef.** 3.1.2 ve 3.1.3'te kapatılan altı temel Cartan ilişkisinden
*türetilebilen* aşağıdaki üç özdeşliği önce **standart** Cartan
calculus'unda, sonra **tilde** (Koszul) tarafında, jenerik test
nesneleri üzerinde sentaktik olarak kapatmak:

$$
\begin{aligned}
& \mathcal{L}_U \mathcal{L}_V \mu - \mathcal{L}_V \mathcal{L}_U \mu - \mathcal{L}_{[U,V]} \mu = 0, \\
& \mathcal{L}_U\, d\,\iota_W \eta - d\,\iota_{[U,W]} \eta - d\,\iota_W\,\mathcal{L}_U \eta = 0, \\
& \mathcal{L}_W\, d\,\iota_V \omega - d\,\iota_W\, d\,\iota_V \omega = 0.
\end{aligned}
$$

Üçü de altı temel ilişkinin sentaktik sonucu:

| # | Standart kanıt zinciri | Tilde kanıt zinciri | Poisson? |
|---|---|---|---|
| 1 | rel-7 ($[\mathcal{L},\mathcal{L}] = \mathcal{L}_{[\cdot]}$) doğrudan | rel-6 ($[\tilde{\mathcal{L}},\tilde{\mathcal{L}}] = \tilde{\mathcal{L}}_{[\cdot]_K}$) doğrudan | tilde: **evet** |
| 2 | rel-5 ($[\mathcal{L},d]=0$) + rel-4 ($[\mathcal{L},\iota] = \iota_{[\cdot]}$) zinciri | aynı zincir tilde'de: rel-5 ($[\tilde{\mathcal{L}},\tilde{d}] = 0$) + rel-4 ($[\tilde{\mathcal{L}},\tilde{\iota}] = \tilde{\iota}_{[\cdot]_K}$) | tilde: **evet** |
| 3 | Cartan magic + $d^2 = 0$ | tilde Cartan magic + $\tilde{d}^2 = 0$ | tilde: **evet** |


## Strateji

**Standart taraf.** 3.1.2'den `engine_of` (1-form intrinsic engine +
3 closure aksiyomu + 3 0-form çöküş kuralı) kullanılıyor;
$\mu, \eta, \omega$ jenerik 1-form, $Y$ jenerik vektör alanı,
ispat `prove_intrinsic_equivalence(LHS, 0, engine=engine_of, …)`.

**Tilde taraf.** 3.1.3'teki `KoszulProblem.tilde_intrinsic_engine()` 28
kuralı + Faz 14.H'de eklenen recognizer iyileştirmesi:
`TildeSnJacobiResidueDefinition` artık iki polariteyi de tanıyor —
canonical (rel-4/rel-6'da görülen) ve overall-sign-flipped (T2/T3
türetilmiş özdeşliklerinin `MultiEval(V, d(·), γ)` sarmalı altında
bıraktığı, tüm işaretleri ters versiyon). Bu sayede üçüncü ve
ikinci özdeşlik V deg-2 üzerinde tek `prob.prove_tilde_cartan(LHS, 0,
etas=(η, ξ))` çağrısıyla kapanıyor.

İlk özdeşlik V 1-VF + tek $\eta$ ile (rel-6 ile birebir aynı kurulum);
ikinci ve üçüncü özdeşlikler V 2-VF + $(\eta, \xi)$ değerlendirmesiyle
kapanıyor — V 1-VF olunca $\tilde{\iota}_\gamma V$ 0-form'a düşüyor
ve `ι̃_γ(d̃ f)` reduction'ı engine'ın doğal kuralları arasında değil.


In [1]:
# Notebook doğrudan açıldığında jacopy'ı import edilebilir hâle getirir.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401


## 1. Kurulum — standart taraf engine + tilde taraf yardımcısı

**Standart semboller:**

- $\mu, \eta, \omega$: jenerik 1-form'lar.
- $U, V, W, Y$: derece-0 vektör alanları (`Derivation`).

**Tilde tarafı:** her özdeşlik için ayrı `KoszulProblem` örneği —
`assume_poisson()` üçü için de gerekli ($[\pi,\pi]_{\mathrm{SN}} = 0$).


In [2]:
# --- standart taraf ---
from jacopy.algebra.derivation import Act, Derivation
from jacopy.algebra.lie_bracket_vf import lie_bracket_vf
from jacopy.calculus.exterior_d import d as default_d
from jacopy.calculus.interior import interior
from jacopy.calculus.intrinsic_engine import (
    intrinsic_engine_with_closure,
    prove_intrinsic_equivalence,
)
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.core.expr import Integer, Neg, Sum, Symbol
from jacopy.core.multi_eval import multi_eval
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.display.jupyter import display_chain
from jacopy.proof.expansion import (
    ExpansionEngine,
    IotaOnZeroFormDefinition,
    IotaSquaredZeroDefinition,
    LieDerivativeOnZeroFormDefinition,
)

reg = PropertyRegistry()
mu = Symbol("μ");    reg.declare(mu, Graded(degree=1))
eta = Symbol("η");   reg.declare(eta, Graded(degree=1))
omega = Symbol("ω"); reg.declare(omega, Graded(degree=1))
U = Derivation("U", 0)
V_vf = Derivation("V", 0)
W = Derivation("W", 0)
Y = Derivation("Y", 0)

defs = list(intrinsic_engine_with_closure().definitions)
defs += [
    IotaOnZeroFormDefinition(registry=reg),
    IotaSquaredZeroDefinition(),
    LieDerivativeOnZeroFormDefinition(registry=reg),
]
engine_of = ExpansionEngine(defs)


def prove_std(label, lhs, rhs):
    chain = prove_intrinsic_equivalence(lhs, rhs, engine=engine_of, registry=reg)
    print(f"{label} → {len(chain)} adımda kapandı")
    return chain


# --- tilde taraf ---
from jacopy.calculus.tilde import (
    TildeExteriorDerivative,
    TildeInteriorProduct,
    TildeLieDerivative,
)
from jacopy.library.koszul_problem import KoszulProblem


def build_tilde(form_names, V_degree=1):
    r = PropertyRegistry()
    pi = Symbol("π")
    forms = tuple(Symbol(n) for n in form_names)
    for f in forms:
        r.declare(f, Graded(degree=1))
    Vmv = Symbol("V")
    r.declare(Vmv, Graded(degree=V_degree))
    prob = KoszulProblem(pi, forms, registry=r, multivectors=((Vmv, V_degree),))
    prob.assume_poisson()
    return (prob, Vmv) + forms


def prove_tilde(label, prob, lhs, rhs, etas):
    chain = prob.prove_tilde_cartan(lhs, rhs, etas=etas)
    print(f"{label} → {len(chain)} adımda kapandı")
    return chain


print("Standart engine kural sayısı :", len(engine_of.definitions))
_probe, *_ = build_tilde(("a", "b"))
print("Tilde engine kural sayısı    :", len(_probe.tilde_intrinsic_engine().definitions))


Standart engine kural sayısı : 15
Tilde engine kural sayısı    : 28


## 2. Özdeşlik 1: $[\mathcal{L}_U, \mathcal{L}_V]\,\mu = \mathcal{L}_{[U,V]}\,\mu$

$$
\mathcal{L}_U \mathcal{L}_V \mu - \mathcal{L}_V \mathcal{L}_U \mu - \mathcal{L}_{[U,V]} \mu \;=\; 0.
$$

Doğrudan rel-7'nin $\mu$ üzerinde değerlendirilmiş hali — 1-form $\mu$
için her üç terim 1-form'dur, $Y$ ile değerlendirilip skalere
düşürülür ve commutator + Jacobi closure aksiyomları üzerinden kapanır.


In [3]:
lhs = Sum.make(
    multi_eval(Act(lie_derivative(U), Act(lie_derivative(V_vf), mu)), Y),
    Neg(multi_eval(Act(lie_derivative(V_vf), Act(lie_derivative(U), mu)), Y)),
    Neg(multi_eval(Act(lie_derivative(lie_bracket_vf(U, V_vf)), mu), Y)),
)
chain = prove_std("(STD-1) (L_U L_V − L_V L_U − L_[U,V]) μ = 0  on μ(Y)",
                  lhs, Integer(0))
display_chain(chain)


(STD-1) (L_U L_V − L_V L_U − L_[U,V]) μ = 0  on μ(Y) → 12 adımda kapandı


\begin{align*}
\left(L_U\!\left(L_V\!\left(\mu\right)\right)\right)\!\left(Y\right) &\to U\!\left(\left(L_V\!\left(\mu\right)\right)\!\left(Y\right)\right) - \left(L_V\!\left(\mu\right)\right)\!\left([U,Y]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(L_V\!\left(\mu\right)\right)\!\left(Y\right) &\to V\!\left(\mu\!\left(Y\right)\right) - \mu\!\left([V,Y]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(L_V\!\left(\mu\right)\right)\!\left([U,Y]_{VF}\right) &\to V\!\left(\mu\!\left([U,Y]_{VF}\right)\right) - \mu\!\left([V,[U,Y]_{VF}]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(L_V\!\left(L_U\!\left(\mu\right)\right)\right)\!\left(Y\right) &\to V\!\left(\left(L_U\!\left(\mu\right)\right)\!\left(Y\right)\right) - \left(L_U\!\left(\mu\right)\right)\!\left([V,Y]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(L_U\!\left(\mu\right)\right)\!\left(Y\right) &\to U\!\left(\mu\!\left(Y\right)\right) - \mu\!\left([U,Y]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(L_U\!\left(\mu\right)\right)\!\left([V,Y]_{VF}\right) &\to U\!\left(\mu\!\left([V,Y]_{VF}\right)\right) - \mu\!\left([U,[V,Y]_{VF}]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(L_[U,V]_{VF}\!\left(\mu\right)\right)\!\left(Y\right) &\to [U,V]_{VF}\!\left(\mu\!\left(Y\right)\right) - \mu\!\left([[U,V]_{VF},Y]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(\left(U\!\left(V\!\left(\mu\!\left(Y\right)\right) - \mu\!\left([V,Y]_{VF}\right)\right) - \left(V\!\left(\mu\!\left([U,Y]_{VF}\right)\right) - \mu\!\left([V,[U,Y]_{VF}]_{VF}\right)\right)\right) - \left(V\!\left(U\!\left(\mu\!\left(Y\right)\right) - \mu\!\left([U,Y]_{VF}\right)\right) - \left(U\!\left(\mu\!\left([V,Y]_{VF}\right)\right) - \mu\!\left([U,[V,Y]_{VF}]_{VF}\right)\right)\right) - \left([U,V]_{VF}\!\left(\mu\!\left(Y\right)\right) -

### 2.T. Tilde analoğu

$$
\tilde{\mathcal{L}}_\alpha \tilde{\mathcal{L}}_\beta V
   - \tilde{\mathcal{L}}_\beta \tilde{\mathcal{L}}_\alpha V
   - \tilde{\mathcal{L}}_{[\alpha,\beta]_K} V \;=\; 0.
$$

Rel-6 (Faz 14.G) ile birebir aynı kurulum: V 1-VF + tek $\eta$
değerlendirmesi, Poisson varsayımı altında 14.G pipeline'ı zaten
mevcut polariteyle tanır.


In [4]:
prob, Vmv, alpha, beta = build_tilde(("α", "β"))
eta_t = Symbol("η"); prob.registry.declare(eta_t, Graded(degree=1))

lhs = Sum.make(
    Act(TildeLieDerivative(alpha, prob.pi),
        Act(TildeLieDerivative(beta, prob.pi), Vmv)),
    Neg(Act(TildeLieDerivative(beta, prob.pi),
            Act(TildeLieDerivative(alpha, prob.pi), Vmv))),
    Neg(Act(TildeLieDerivative(prob.bracket(alpha, beta), prob.pi), Vmv)),
)
chain = prove_tilde(
    "(TIL-1) (L̃_α L̃_β − L̃_β L̃_α − L̃_[α,β]_K) V = 0  [Poisson]",
    prob, lhs, Integer(0), etas=(eta_t,))
display_chain(chain)


(TIL-1) (L̃_α L̃_β − L̃_β L̃_α − L̃_[α,β]_K) V = 0  [Poisson] → 120 adımda kapandı


\begin{align*}
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right) - \tilde{\mathcal{L}}_{\beta}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right) - \tilde{\mathcal{L}}_{\left[\alpha,\, \beta\right]_{[\cdot,\cdot]_K[\pi]}}\!\left(V\right)\right)\!\left(\eta\right) &\to \left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta\right) - \left(\tilde{\mathcal{L}}_{\beta}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\!\left(\eta\right) - \left(\tilde{\mathcal{L}}_{\left[\alpha,\, \beta\right]_{[\cdot,\cdot]_K[\pi]}}\!\left(V\right)\right)\!\left(\eta\right) && \text{[MultiEval head linearity]\,(axiom)}\;\text{--- apply axiom: MultiEval head linearity} \\
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta\right) &\to \left(\pi\sharp\!\left(\alpha\right)\right)\!\left(\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\!\left(\eta\right)\right) - \left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\!\left(\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]}\right) && \text{[L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)]\,(axiom)}\;\text{--- apply axiom: L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)} \\
\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\!\left(\eta\right) &\to \left(\pi\sharp\!\left(\beta\right)\right)\!\left(V\!\left(\eta\right)\right) - V\!\left(\left[\beta,\, \eta\right]_{[\cdot,\cdot]_K[\pi]}\right) && \text{[L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)]\,(axiom)}\;\text{--- apply axiom: L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)} \\
\left[\beta,\, \eta\right]_{[\cdot,\cdot]_K[\pi]} &\to L_\pi\sharp(\beta)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\beta\right) - d\!\left(\langle \pi\sharp\!\left(\beta\right),\, \eta \rangle\right) && \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) − L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) − d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)}\;\text{--- apply axiom: [\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) − L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) − d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]} \\
V\!\left(L_\pi\sharp(\beta)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\beta\right) - d\!\left(\langle \pi\sharp\!\left(\beta\right),\, \eta \rangle\right)\right) &\to V\!\left(L_\pi\sharp(\beta)\!\left(\eta\right)\right) - V\!\left(L_\pi\sharp(\eta)\!\left(\beta\right)\right) - V\!\left(d\!\left(\langle \pi\sharp\!\left(\beta\right),\, \eta \

## 3. Özdeşlik 2: $\mathcal{L}_U\,d\,\iota_W\,\eta = d\,\iota_{[U,W]}\,\eta + d\,\iota_W\,\mathcal{L}_U\,\eta$

$$
\mathcal{L}_U\, d\,\iota_W \eta - d\,\iota_{[U,W]} \eta - d\,\iota_W\,\mathcal{L}_U \eta \;=\; 0.
$$

Türetme: $[\mathcal{L}_U, d] = 0$ (rel-5) ile $\mathcal{L}_U\,d = d\,\mathcal{L}_U$,
ardından $[\mathcal{L}_U, \iota_W] = \iota_{[U,W]}$ (rel-4) ile
$\mathcal{L}_U\,\iota_W \eta = \iota_{[U,W]} \eta + \iota_W\,\mathcal{L}_U \eta$.
$d$ uygulanınca eşitlik çıkar. 1-form $\eta$, $Y$ ile değerlendirildiğinde
her üç terim 1-form ve skalere düşer.


In [5]:
lhs = Sum.make(
    multi_eval(Act(lie_derivative(U), Act(default_d, Act(interior(W), eta))), Y),
    Neg(multi_eval(Act(default_d, Act(interior(lie_bracket_vf(U, W)), eta)), Y)),
    Neg(multi_eval(Act(default_d, Act(interior(W), Act(lie_derivative(U), eta))), Y)),
)
chain = prove_std(
    "(STD-2) L_U d ι_W η − d ι_[U,W] η − d ι_W L_U η = 0  on η(Y)",
    lhs, Integer(0))
display_chain(chain)


(STD-2) L_U d ι_W η − d ι_[U,W] η − d ι_W L_U η = 0  on η(Y) → 14 adımda kapandı


\begin{align*}
\left(L_U\!\left(d\!\left(\iota_W\!\left(\eta\right)\right)\right)\right)\!\left(Y\right) &\to U\!\left(\left(d\!\left(\iota_W\!\left(\eta\right)\right)\right)\!\left(Y\right)\right) - \left(d\!\left(\iota_W\!\left(\eta\right)\right)\right)\!\left([U,Y]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(d\!\left(\iota_W\!\left(\eta\right)\right)\right)\!\left(Y\right) &\to Y\!\left(\iota_W\!\left(\eta\right)\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
Y\!\left(\iota_W\!\left(\eta\right)\right) &\to Y\!\left(\eta\!\left(W\right)\right) && \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) → Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)}\;\text{--- apply axiom: bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) → Act(D, MultiEval(\ensuremath{\omega}, X))} \\
\left(d\!\left(\iota_W\!\left(\eta\right)\right)\right)\!\left([U,Y]_{VF}\right) &\to [U,Y]_{VF}\!\left(\iota_W\!\left(\eta\right)\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
[U,Y]_{VF}\!\left(\iota_W\!\left(\eta\right)\right) &\to [U,Y]_{VF}\!\left(\eta\!\left(W\right)\right) && \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) → Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)}\;\text{--- apply axiom: bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) → Act(D, MultiEval(\ensuremath{\omega}, X))} \\
\left(d\!\left(\iota_[U,W]_{VF}\!\left(\eta\right)\right)\right)\!\left(Y\right) &\to Y\!\left(\iota_[U,W]_{VF}\!\left(\eta\right)\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
Y\!\left(\iota_[U,W]_{VF}\!\left(\eta\right)\right) &\to Y\!\left(\eta\!\left([U,W]_{VF}\right)\right) && \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) → Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)}\;\text{--- apply axiom: bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) → Act(D, MultiEval(\ensuremath{\omega}, X))} \\
\left(d\!\left(\iota_W\!\left(L_U\!\left(\eta\right)\right)\right)\right)\!\left(Y\right) &\to Y\!\left(\iota_W\!\left(L_U\!\left(\eta\right)\right)\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
Y\!\left(\iota_W\!\left(L_U\!\left(\eta\right)\right)\

### 3.T. Tilde analoğu

$$
\tilde{\mathcal{L}}_\alpha\,\tilde{d}\,\tilde{\iota}_\gamma V
   - \tilde{d}\,\tilde{\iota}_{[\alpha,\gamma]_K} V
   - \tilde{d}\,\tilde{\iota}_\gamma\,\tilde{\mathcal{L}}_\alpha V \;=\; 0.
$$

V'yi **2-VF** alıyoruz; $\tilde{\iota}_\gamma V$ 1-VF, $\tilde{d}$ ile
2-VF olur, $\tilde{\mathcal{L}}_\alpha$ koruyup yine 2-VF bırakır.
$(\eta, \xi)$ değerlendirmesi her üç terimi fonksiyona indirir.
Engine $\tilde{d}$-altında sarmalanmış `MultiEval(V, d(⟨...⟩), γ)`
formundaki SN-Jacobi engelini, **Faz 14.H** polarity-flip
genişlemesi sayesinde tanır ve sıfırlar.


In [6]:
prob, Vmv, alpha, gamma = build_tilde(("α", "γ"), V_degree=2)
eta_t = Symbol("η"); prob.registry.declare(eta_t, Graded(degree=1))
xi_t = Symbol("ξ");  prob.registry.declare(xi_t, Graded(degree=1))
dt = TildeExteriorDerivative(prob.pi)

lhs = Sum.make(
    Act(TildeLieDerivative(alpha, prob.pi),
        Act(dt, Act(TildeInteriorProduct(gamma), Vmv))),
    Neg(Act(dt,
        Act(TildeInteriorProduct(prob.bracket(alpha, gamma)), Vmv))),
    Neg(Act(dt,
        Act(TildeInteriorProduct(gamma),
            Act(TildeLieDerivative(alpha, prob.pi), Vmv)))),
)
chain = prove_tilde(
    "(TIL-2) L̃_α d̃ ι̃_γ V − d̃ ι̃_[α,γ]_K V − d̃ ι̃_γ L̃_α V = 0  [Poisson]",
    prob, lhs, Integer(0), etas=(eta_t, xi_t))
display_chain(chain)


(TIL-2) L̃_α d̃ ι̃_γ V − d̃ ι̃_[α,γ]_K V − d̃ ι̃_γ L̃_α V = 0  [Poisson] → 199 adımda kapandı


\begin{align*}
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\right) - \tilde{d}\!\left(\tilde{\iota}_{\left[\alpha,\, \gamma\right]_{[\cdot,\cdot]_K[\pi]}}\!\left(V\right)\right) - \tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\right)\!\left(\eta,\, \xi\right) &\to \left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\right)\right)\!\left(\eta,\, \xi\right) - \left(\tilde{d}\!\left(\tilde{\iota}_{\left[\alpha,\, \gamma\right]_{[\cdot,\cdot]_K[\pi]}}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right) - \left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\right)\!\left(\eta,\, \xi\right) && \text{[MultiEval head linearity]\,(axiom)}\;\text{--- apply axiom: MultiEval head linearity} \\
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\right)\right)\!\left(\eta,\, \xi\right) &\to \left(\pi\sharp\!\left(\alpha\right)\right)\!\left(\left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right)\right) - \left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\right)\!\left(\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]},\, \xi\right) - \left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\right)\!\left(\eta,\, \left[\alpha,\, \xi\right]_{[\cdot,\cdot]_K[\pi]}\right) && \text{[L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)]\,(axiom)}\;\text{--- apply axiom: L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)} \\
\left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right) &\to \left(\pi\sharp\!\left(\eta\right)\right)\!\left(\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\!\left(\xi\right)\right) - \left(\pi\sharp\!\left(\xi\right)\right)\!\left(\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\!\left(\eta\right)\right) - \left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\!\left(\left[\eta,\, \xi\right]_{[\cdot,\cdot]_K[\pi]}\right) && \text{[d̃ intrinsic (Koszul) [\ensuremath{\pi}]: (d̃V)(\ensuremath{\eta}\_0, …) = \ensuremath{\Sigma} ±\ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\eta}\_i)\ensuremath{\cdot}V(…) + \ensuremath{\Sigma} ±V([\ensuremath{\eta}\_i,\ensuremath{\eta}\_j]\_K, …)]\,(axiom)}\;\text{--- apply axiom: d̃ intrinsic (Koszul) [\ensuremath{\pi}]: (d̃V)(\ensuremath{\eta}\_0, …) = \ensuremath{\Sigma} ±\ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\eta}\_i)\ensuremath{\cdot}V(…) + \ensuremath{\Sigma} ±V([\ensuremath{\eta}\_i,\ensuremath{\eta}\_j]\_K, …)} \\
\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\!\left(\xi\right) &\to V\!\left(\gamma,\, \xi\right) && \text{[\ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)} \\
\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\!\left(\eta\right) &\to V\!\left(\gamma,\, \eta\right) && \text{[\ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, 

## 4. Özdeşlik 3: $\mathcal{L}_W\,d\,\iota_V \omega = d\,\iota_W\,d\,\iota_V \omega$

$$
\mathcal{L}_W\, d\,\iota_V \omega - d\,\iota_W\, d\,\iota_V \omega \;=\; 0.
$$

Türetme: Cartan magic ile $\mathcal{L}_W = d\,\iota_W + \iota_W\, d$,
$A := d\,\iota_V \omega$ olarak alındığında
$\mathcal{L}_W A = d\,\iota_W A + \iota_W\,d\,A = d\,\iota_W\,d\,\iota_V \omega
+ \iota_W\,d^2\,\iota_V \omega = d\,\iota_W\,d\,\iota_V \omega + 0$.
Yani özdeşlik aslında $\iota_W\,d^2 = 0$'a dayanır.


In [7]:
lhs = Sum.make(
    multi_eval(Act(lie_derivative(W), Act(default_d, Act(interior(V_vf), omega))), Y),
    Neg(multi_eval(Act(default_d, Act(interior(W),
                Act(default_d, Act(interior(V_vf), omega)))), Y)),
)
chain = prove_std(
    "(STD-3) L_W d ι_V ω − d ι_W d ι_V ω = 0  on ω(Y)",
    lhs, Integer(0))
display_chain(chain)


(STD-3) L_W d ι_V ω − d ι_W d ι_V ω = 0  on ω(Y) → 12 adımda kapandı


\begin{align*}
\left(L_W\!\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\right)\!\left(Y\right) &\to W\!\left(\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\!\left(Y\right)\right) - \left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\!\left([W,Y]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\!\left(Y\right) &\to Y\!\left(\iota_V\!\left(\omega\right)\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
Y\!\left(\iota_V\!\left(\omega\right)\right) &\to Y\!\left(\omega\!\left(V\right)\right) && \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) → Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)}\;\text{--- apply axiom: bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) → Act(D, MultiEval(\ensuremath{\omega}, X))} \\
\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\!\left([W,Y]_{VF}\right) &\to [W,Y]_{VF}\!\left(\iota_V\!\left(\omega\right)\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
[W,Y]_{VF}\!\left(\iota_V\!\left(\omega\right)\right) &\to [W,Y]_{VF}\!\left(\omega\!\left(V\right)\right) && \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) → Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)}\;\text{--- apply axiom: bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) → Act(D, MultiEval(\ensuremath{\omega}, X))} \\
\left(d\!\left(\iota_W\!\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\right)\right)\!\left(Y\right) &\to Y\!\left(\iota_W\!\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
Y\!\left(\iota_W\!\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\right) &\to Y\!\left(\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\!\left(W\right)\right) && \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) → Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)}\;\text{--- apply axiom: bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) → Act(D, MultiEval(\ensuremath{\omega}, X))} \\
\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\!\left(W\right) &\to W\!\left(\iota_V\!\left(\omega\right)\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_

### 4.T. Tilde analoğu

$$
\tilde{\mathcal{L}}_\gamma\,\tilde{d}\,\tilde{\iota}_\beta V
   - \tilde{d}\,\tilde{\iota}_\gamma\,\tilde{d}\,\tilde{\iota}_\beta V \;=\; 0.
$$

Aynı kurulum (V 2-VF, $(\eta, \xi)$ değerlendirme): hem $\tilde{d}^2 = 0$
(Aux-5, Poisson) hem tilde Cartan magic (14.F) hem de 14.G+H pipeline'ı
gerekir. Residue $\tilde{d}$-sarmalı altında $V(d(⟨...⟩), \beta)$
formuna düşer, Faz 14.H polarity-flip recognizer'ı kapatır.


In [8]:
prob, Vmv, beta, gamma = build_tilde(("β", "γ"), V_degree=2)
eta_t = Symbol("η"); prob.registry.declare(eta_t, Graded(degree=1))
xi_t = Symbol("ξ");  prob.registry.declare(xi_t, Graded(degree=1))
dt = TildeExteriorDerivative(prob.pi)

lhs = Sum.make(
    Act(TildeLieDerivative(gamma, prob.pi),
        Act(dt, Act(TildeInteriorProduct(beta), Vmv))),
    Neg(Act(dt, Act(TildeInteriorProduct(gamma),
            Act(dt, Act(TildeInteriorProduct(beta), Vmv))))),
)
chain = prove_tilde(
    "(TIL-3) L̃_γ d̃ ι̃_β V − d̃ ι̃_γ d̃ ι̃_β V = 0  [Poisson]",
    prob, lhs, Integer(0), etas=(eta_t, xi_t))
display_chain(chain)


(TIL-3) L̃_γ d̃ ι̃_β V − d̃ ι̃_γ d̃ ι̃_β V = 0  [Poisson] → 190 adımda kapandı


\begin{align*}
\left(\tilde{\mathcal{L}}_{\gamma}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right) - \tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\right)\right)\!\left(\eta,\, \xi\right) &\to \left(\tilde{\mathcal{L}}_{\gamma}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\right)\!\left(\eta,\, \xi\right) - \left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\right)\right)\!\left(\eta,\, \xi\right) && \text{[MultiEval head linearity]\,(axiom)}\;\text{--- apply axiom: MultiEval head linearity} \\
\left(\tilde{\mathcal{L}}_{\gamma}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\right)\!\left(\eta,\, \xi\right) &\to \left(\pi\sharp\!\left(\gamma\right)\right)\!\left(\left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right)\right) - \left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\!\left(\left[\gamma,\, \eta\right]_{[\cdot,\cdot]_K[\pi]},\, \xi\right) - \left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta,\, \left[\gamma,\, \xi\right]_{[\cdot,\cdot]_K[\pi]}\right) && \text{[L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)]\,(axiom)}\;\text{--- apply axiom: L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)} \\
\left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right) &\to \left(\pi\sharp\!\left(\eta\right)\right)\!\left(\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\xi\right)\right) - \left(\pi\sharp\!\left(\xi\right)\right)\!\left(\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\eta\right)\right) - \left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\left[\eta,\, \xi\right]_{[\cdot,\cdot]_K[\pi]}\right) && \text{[d̃ intrinsic (Koszul) [\ensuremath{\pi}]: (d̃V)(\ensuremath{\eta}\_0, …) = \ensuremath{\Sigma} ±\ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\eta}\_i)\ensuremath{\cdot}V(…) + \ensuremath{\Sigma} ±V([\ensuremath{\eta}\_i,\ensuremath{\eta}\_j]\_K, …)]\,(axiom)}\;\text{--- apply axiom: d̃ intrinsic (Koszul) [\ensuremath{\pi}]: (d̃V)(\ensuremath{\eta}\_0, …) = \ensuremath{\Sigma} ±\ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\eta}\_i)\ensuremath{\cdot}V(…) + \ensuremath{\Sigma} ±V([\ensuremath{\eta}\_i,\ensuremath{\eta}\_j]\_K, …)} \\
\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\xi\right) &\to V\!\left(\beta,\, \xi\right) && \text{[\ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)} \\
\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\eta\right) &\to V\!\left(\beta,\, \eta\right) && \text{[\ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)} \\
\left

## Sonuç

Üç türetilmiş özdeşliğin standart ve tilde sürümleri tek
`prove_intrinsic_equivalence` / `prob.prove_tilde_cartan` çağrısıyla
syntactic olarak kapandı. Tilde tarafında ikinci ve üçüncü
özdeşliğin kapanması **Faz 14.H** ile gelen polarity-flip
genişlemesini gerektirdi: `TildeSnJacobiResidueDefinition` artık
canonical residue'yu da, overall-sign-flipped versiyonu da tanıyor.

| # | Standart | Tilde | $V$ | Eval | Poisson |
|---|---|---|---|---|---|
| 1 | $[\mathcal{L},\mathcal{L}]\mu = \mathcal{L}_{[\cdot]}\mu$, $\mu(Y)$ | $[\tilde{\mathcal{L}},\tilde{\mathcal{L}}] V = \tilde{\mathcal{L}}_{[\cdot]_K} V$ | 1-VF | $(\eta)$ | tilde |
| 2 | $\mathcal{L}\,d\iota - d\iota_{[\cdot]} - d\iota\,\mathcal{L}$ on $\eta(Y)$ | aynı, tilde | 2-VF | $(\eta, \xi)$ | tilde |
| 3 | $\mathcal{L}\,d\iota - d\iota\,d\iota$ on $\omega(Y)$ | aynı, tilde | 2-VF | $(\eta, \xi)$ | tilde |

İspatların tamamı `display_chain` ile LaTeX olarak okunabilir;
her adımın hangi `Definition`'dan geldiği `ProofStep.rule` alanında.
